In [ ]:
"""
DL_QA
Versiyon: 27.0
"""

import sys, os, time, logging, threading, shutil
from datetime import datetime
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import numpy as np
import cv2
from PIL import Image, ImageTk
import snap7
import random
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler
import queue
import gc

class NullWriter:
    def write(self, t): pass
    def flush(self): pass
    def isatty(self): return False
if sys.stdout is None: sys.stdout = NullWriter()
if sys.stderr is None: sys.stderr = NullWriter()

tf = None
DEFAULT_MODEL_NAME = 'dokum_hata_tespit_modeli.keras'
DEFAULT_MODEL_NAME_MOBILE = 'dokum_hata_tespit_modeli_MobileNet.keras'

IMG_SIZE           = (300, 300)
SPLASH_IMAGE_NAME  = 'splash.png'
LOG_FILE           = 'kalite_kontrol.log'
PLC_IP             = "192.168.8.167"
PLC_RACK, PLC_SLOT = 0, 1
DB_NUM             = 1
OFFSET_CH1, OFFSET_CH2 = 0, 2

def loglama_kur():
    logger = logging.getLogger("MKE_AI")
    logger.setLevel(logging.DEBUG)
    for h in logger.handlers[:]:
        try: h.close()
        except Exception: pass
        logger.removeHandler(h)
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s",
                            datefmt="%Y-%m-%d %H:%M:%S")
    try:
        fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
        fh.setLevel(logging.DEBUG); fh.setFormatter(fmt); logger.addHandler(fh)
    except Exception: pass
    try:
        if sys.stdout and not isinstance(sys.stdout, NullWriter):
            ch = logging.StreamHandler(sys.stdout)
            ch.setLevel(logging.INFO); ch.setFormatter(fmt); logger.addHandler(ch)
    except Exception: pass
    return logger

log = loglama_kur()

class PLCYoneticisi:
    def __init__(self):
        self.client = snap7.client.Client()
        self.connected = False
        self.lock = threading.Lock()
        self.simulasyon_modu = False
        self.ip = PLC_IP; self.rack = PLC_RACK
        self.slot = PLC_SLOT; self.db_num = DB_NUM
        
    def baglan(self):
        if self.simulasyon_modu: return False
        try:
            self.client.connect(self.ip, self.rack, self.slot)
            self.connected = self.client.get_connected()
            if self.connected:
                log.info(f"PLC baglandi -> {self.ip} Rack={self.rack} Slot={self.slot}")
            return self.connected
        except Exception as e:
            log.warning(f"PLC baglanti hatasi: {e} -- Simulasyon aktif.")
            self.simulasyon_modu = True; self.connected = False; return False

    def yaz(self, deger, offset):
        if self.simulasyon_modu:
            log.debug(f"[SIM] PLC yaz: offset={offset} deger={deger}"); return
        with self.lock:
            if not self.connected:
                if not self.baglan(): return
            try:
                data = bytearray(2); snap7.util.set_int(data, 0, deger)
                self.client.db_write(self.db_num, offset, data)              
                log.debug(f"PLC yazildi: DB{self.db_num} offset={offset} deger={deger}")
            except Exception as e:
                log.error(f"PLC yazma hatasi: {e}")
                self.connected = False; self.simulasyon_modu = True

    def yaz_ve_onayla(self, karar_degeri, karar_offset, ack_byte_offset, ack_bit_index):
        if self.simulasyon_modu:
            self.yaz(karar_degeri, karar_offset)
            time.sleep(0.15)
            self.yaz(0, karar_offset)
            return True

        self.yaz(karar_degeri, karar_offset)
        
        baslangic = time.perf_counter()
        while True:
            if time.perf_counter() - baslangic > 1.0: 
                log.error(f"[TIMEOUT] PLC veriyi okumadi! Offset: {karar_offset}")
                return False
                
            with self.lock:
                data = self.client.db_read(self.db_num, ack_byte_offset, 1)
            ack_durumu = snap7.util.get_bool(data, 0, ack_bit_index)
            
            if ack_durumu:
                break
            time.sleep(0.01)

        self.yaz(0, karar_offset)
        
        baslangic = time.perf_counter()
        while True:
            if time.perf_counter() - baslangic > 1.0:
                log.error(f"[TIMEOUT] PLC ACK bitini sifirlamadi! Offset: {karar_offset}")
                return False
                
            with self.lock:
                data = self.client.db_read(self.db_num, ack_byte_offset, 1)
            ack_durumu = snap7.util.get_bool(data, 0, ack_bit_index)
            
            if not ack_durumu:
                break
            time.sleep(0.01)

        return True

class DosyaYakalayici(FileSystemEventHandler):
    def __init__(self, app, channel_id):
        self.app = app; self.channel_id = channel_id

    def on_created(self, event):
        if not self.app.is_running: return
        
        if (not event.is_directory and
            event.src_path.lower().endswith(('.png','.jpg','.jpeg','.bmp'))):
            threading.Thread(target=self._dosyayi_bekle_ve_ilet, args=(event.src_path, self.channel_id), daemon=True).start()

    def _dosyayi_bekle_ve_ilet(self, src_path, channel_id):
        for _ in range(20): 
            try:                
                with open(src_path, 'a'): 
                    pass          
                
                if self.app.is_running:
                    threading.Thread(target=self.app.dosya_analiz, args=(src_path, channel_id), daemon=True).start()
                return                
            except PermissionError:
                time.sleep(0.05) 
            except Exception:
                break 
                
        log.error(f"Dosya okuma hatasi (kilitli kaldi): {src_path}")

class KaliteKontrolApp:
    def __init__(self, root):
        self.root = root
        self.stats = {1: {'ok': 0, 'nok': 0}, 2: {'ok': 0, 'nok': 0}}
        self.e2e_gecmisi = {1: [], 2: []}
        self.root.title("MKE - AI Vision v27.0")
        self.root.geometry("1280x960")
        self.is_running = True
        self.root.protocol("WM_DELETE_WINDOW", self.on_closing)

        self.plc = PLCYoneticisi()
        self.model = None
        self.last_conv_layer_name = None
        self.model_lock = threading.Lock()
        self.observer1 = self.observer2 = None

        self.base_dir    = os.getcwd()
        self.dir_ch1_in  = os.path.join(self.base_dir, "KAMERA_1_GELEN")
        self.dir_ch1_ok  = os.path.join(self.base_dir, "KAMERA_1_ARSIV_OK")
        self.dir_ch1_nok = os.path.join(self.base_dir, "KAMERA_1_ARSIV_NOK")
        self.dir_ch2_in  = os.path.join(self.base_dir, "KAMERA_2_GELEN")
        self.dir_ch2_ok  = os.path.join(self.base_dir, "KAMERA_2_ARSIV_OK")
        self.dir_ch2_nok = os.path.join(self.base_dir, "KAMERA_2_ARSIV_NOK")
        for d in [self.dir_ch1_in, self.dir_ch1_ok, self.dir_ch1_nok,
                  self.dir_ch2_in, self.dir_ch2_ok, self.dir_ch2_nok]:
            os.makedirs(d, exist_ok=True)

        self.arsiv_limiti  = tk.IntVar(value=50)
        self.guven_esigi   = tk.DoubleVar(value=0.50)
        self.epoch_sayisi  = tk.IntVar(value=5)
        self.learning_rate = tk.DoubleVar(value=1e-5)
        self.batch_size    = tk.IntVar(value=4)
        self.plc_ip         = tk.StringVar(value=PLC_IP)
        self.plc_rack       = tk.IntVar(value=0)
        self.plc_slot       = tk.IntVar(value=1)
        self.plc_db         = tk.IntVar(value=DB_NUM)
        self.plc_offset_ch1 = tk.IntVar(value=OFFSET_CH1)
        self.plc_offset_ch2 = tk.IntVar(value=OFFSET_CH2)

        log.info("Uygulama baslatildi.")
        self.setup_ui()
        self.islem_kuyrugu = queue.Queue() 
        self._kuyruk_dinleyici() 
        self.root.bind("<F12>", self._test_cakisma)
        self.check_model()
        self.plc_yeniden_baglan(sessiz=True)
        
    def holdout_test_baslat(self):
        if not self.model:
            messagebox.showerror("Hata", "Önce model yükleyin.")
            return
    
        klasor = filedialog.askdirectory(title="Holdout Test Klasörünü Seçin (OK/NOK alt klasörleri olmalı)")
        if not klasor: return
        gc.collect()
        threading.Thread(target=self._holdout_test_calistir, args=(klasor,), daemon=True).start()
    def _holdout_test_calistir(self, klasor):
        try:
            test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)
            
            gen = test_datagen.flow_from_directory(
                klasor,
                target_size=IMG_SIZE,
                batch_size=16,
                class_mode='binary',
                shuffle=False
            )
            if gen.samples == 0:
                self.root.after(0, lambda: messagebox.showerror("Hata", "Klasörde görüntü bulunamadı."))
                return
            with self.model_lock:
                tahminler = self.model.predict(gen, verbose=0)
            esik     = self.guven_esigi.get()
            idx_ok   = gen.class_indices.get('OK',  gen.class_indices.get('ok',  1))
            idx_nok  = gen.class_indices.get('NOK', gen.class_indices.get('nok', 0))
            
            # Tahmin: score < esik → NOK (1), değilse OK (0)
            kararlar  = (tahminler[:, 0] < esik).astype(int)
            gercekler = (gen.classes == idx_nok).astype(int)
            
            tp = int(((kararlar == 1) & (gercekler == 1)).sum())
            tn = int(((kararlar == 0) & (gercekler == 0)).sum())
            fp = int(((kararlar == 1) & (gercekler == 0)).sum())
            fn = int(((kararlar == 0) & (gercekler == 1)).sum())
            
            toplam    = tp + tn + fp + fn
            accuracy  = (tp + tn) / toplam        if toplam > 0  else 0
            precision = tp / (tp + fp)            if (tp + fp) > 0 else 0
            recall    = tp / (tp + fn)            if (tp + fn) > 0 else 0
            f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
            fn_oran   = fn / (tp + fn)            if (tp + fn) > 0 else 0

            sonuc = (
                f"{'='*45}\n"
                f"  HOLDOUT TEST SONUÇLARI\n"
                f"{'='*45}\n"
                f"  Toplam Görüntü : {toplam}\n"
                f"  Eşik Değeri    : {esik:.2f}\n"
                f"{'-'*45}\n"
                f"  Doğruluk       : %{accuracy*100:.1f}\n"
                f"  Kesinlik       : %{precision*100:.1f}\n"
                f"  Duyarlılık     : %{recall*100:.1f}\n"
                f"  F1 Skoru       : %{f1*100:.1f}\n"
                f"  False Negative : %{fn_oran*100:.1f}\n"
                f"{'-'*45}\n"
                f"  TP: {tp}  TN: {tn}  FP: {fp}  FN: {fn}\n"
                f"{'='*45}"
            )
        
            log.info(f"Holdout test tamamlandi:\n{sonuc}")
            self.root.after(0, lambda s=sonuc: messagebox.showinfo("Holdout Test Sonucu", s))
        except Exception as e:
            
            log.error(f"Holdout test hatasi: {e}", exc_info=True)
            self.root.after(0, lambda: messagebox.showerror("Hata", str(e)))
        
    def _kuyruk_dinleyici(self):
        try:
            if not self.islem_kuyrugu.empty():
                src_path, channel_id = self.islem_kuyrugu.get_nowait()
                self.dosya_analiz(src_path, channel_id)
        except Exception as e:
            log.error(f"Kuyruk isleme hatasi: {e}")
        finally:
            if self.is_running:
                self.root.after(50, self._kuyruk_dinleyici)

    def on_closing(self):
        try:
            self.is_running = False
            if hasattr(self, '_ui_log_handler'):
                try: logging.getLogger("MKE_AI").removeHandler(self._ui_log_handler)
                except Exception: pass
            if self.observer1: self.observer1.stop(); self.observer1.join(timeout=1.0)
            if self.observer2: self.observer2.stop(); self.observer2.join(timeout=1.0)
            self.root.destroy()
        except Exception: pass

    # --- MODEL ---
    def check_model(self):
        for name in [DEFAULT_MODEL_NAME_MOBILE, DEFAULT_MODEL_NAME]:
            p = os.path.join(os.getcwd(), name)
            if os.path.exists(p):
                self.load_model(p)
                break  

    def _modeli_sifirdan_kur(self):
        if not tf:
            messagebox.showerror("Hata", "TensorFlow yuklenmedi.")
            return
            
        cevap = messagebox.askyesno("Onay", "Mevcut model silinip MobileNetV2 kurulacak. Onayliyor musunuz?")
        if not cevap: return

        with self.model_lock:
            base = tf.keras.applications.MobileNetV2(
                input_shape=(300, 300, 3),
                include_top=False,
                weights='imagenet'
            )
            base.trainable = False 

            self.model = tf.keras.Sequential([
                base,
                tf.keras.layers.GlobalAveragePooling2D(),
                tf.keras.layers.Dense(128, activation='relu'),
                tf.keras.layers.Dropout(0.4),
                tf.keras.layers.Dense(1, activation='sigmoid')
            ])

            self.last_conv_layer_name = self._find_last_conv_layer()

        self.lbl_model.config(text="Model: MobileNetV2 (Transfer Learning)", fg="blue")
        log.info(f"MobileNetV2 kuruldu. Grad-CAM katmani: {self.last_conv_layer_name}")
        messagebox.showinfo("Basarili", "MobileNetV2 hazir.\nEgitimi baslatabilirsiniz.")

    def load_model(self, p):
        if not tf:
            messagebox.showerror("Hata", "TensorFlow yuklenmedi."); return
        try:
            self.model = tf.keras.models.load_model(p)
            self.lbl_model.config(text=f"Model: {os.path.basename(p)}", fg="green")
            self.last_conv_layer_name = self._find_last_conv_layer()
            log.info(f"Model yuklendi: {p} | Grad-CAM: {self.last_conv_layer_name}")
        except Exception as e:
            log.error(f"Model yukleme hatasi: {e}")
            messagebox.showerror("Model Hatasi", f"Model yuklenemedi:\n{e}")

    def browse_model(self):
        p = filedialog.askopenfilename(filetypes=[("Keras Model", "*.keras")])
        if p: self.load_model(p)

    # --- GRAD-CAM ---
    def _find_last_conv_layer(self):
        CONV = frozenset(['Conv2D','DepthwiseConv2D','SeparableConv2D','Conv2DTranspose'])
        bulunanlar = []
        def _tara(katmanlar):
            for layer in katmanlar:
                if type(layer).__name__ in CONV:
                    bulunanlar.append(layer.name)
                alt = getattr(layer, 'layers', None)
                if alt: _tara(alt)
        _tara(self.model.layers)
        if not bulunanlar:
            for layer in self.model.layers:
                if hasattr(layer, 'kernel_size') or hasattr(layer, 'filters'):
                    bulunanlar.append(layer.name)
        if bulunanlar:
            sec = bulunanlar[-1]
            log.info(f"Grad-CAM: {len(bulunanlar)} Conv2D bulundu, secilen: '{sec}'")
            return sec
        log.warning("Grad-CAM: Conv2D bulunamadi -- devre disi.")
        return None

    def _make_gradcam_heatmap(self, img_array):
        if not self.last_conv_layer_name: return None
        try:
            img_tensor = tf.cast(img_array, tf.float32)
    
            base_submodel = None
            head_layers = []
            
            for layer in self.model.layers:
                if hasattr(layer, 'layers') and len(getattr(layer, 'layers', [])) > 10:
                    base_submodel = layer
                elif base_submodel is not None:
                    head_layers.append(layer)
    
            if base_submodel is not None:
                try:
                    last_conv_layer = base_submodel.get_layer(self.last_conv_layer_name)
                except ValueError:
                    return None
    
                last_conv_model = tf.keras.Model(base_submodel.input, last_conv_layer.output)
                
                layer_names = [l.name for l in base_submodel.layers]
                idx = layer_names.index(self.last_conv_layer_name)
                post_conv_layers = base_submodel.layers[idx+1:]
    
                with tf.GradientTape() as tape:
                    conv_out = last_conv_model(img_tensor, training=False)
                    tape.watch(conv_out)
                    
                    x = conv_out
                    for layer in post_conv_layers:
                        x = layer(x, training=False)
                        
                    for layer in head_layers:
                        if layer == head_layers[-1] and hasattr(layer, 'activation') and layer.activation.__name__ == 'sigmoid':
                            x = tf.matmul(x, layer.kernel) + layer.bias
                        else:
                            x = layer(x, training=False)
                            
                    logit = x[:, 0]
                    score = logit if logit > 0 else -logit
    
                grads = tape.gradient(score, conv_out)
                if grads is None: return None
    
                pw = tf.reduce_mean(grads[0], axis=(0, 1))
                hm = tf.reduce_sum(conv_out[0] * pw, axis=-1)
    
            else:
                before, after, found = [], [], False
                for layer in self.model.layers:
                    if found: after.append(layer)
                    else:
                        before.append(layer)
                        if layer.name == self.last_conv_layer_name: found = True
                if not found: return None
    
                x = img_tensor
                with tf.GradientTape() as tape2:
                    with tf.GradientTape() as tape1:
                        with tf.GradientTape() as tape0:
                            for layer in before: x = layer(x, training=False)
                            conv_out = x
                            tape0.watch(conv_out)
                            tape1.watch(conv_out)
                            tape2.watch(conv_out)
                            for layer in after: x = layer(x, training=False)
                            score = 1.0 - x[:, 0]
                        grads1 = tape0.gradient(score, conv_out)
                    grads2 = tape1.gradient(grads1, conv_out)
                grads3 = tape2.gradient(grads2, conv_out)
    
                if grads1 is None: return None
                nom   = grads2[0]
                denom = 2.0 * grads2[0] + conv_out[0] * grads3[0] + 1e-7
                alpha = tf.nn.relu(nom / denom)
                pw    = tf.reduce_sum(alpha * tf.nn.relu(grads1[0]), axis=(0, 1))
                hm    = tf.reduce_sum(conv_out[0] * pw, axis=-1)
    
            hm = tf.maximum(hm, 0)
            mx = tf.math.reduce_max(hm)
            if mx == 0: return None
            return (hm / mx).numpy()

        except Exception as e:
            log.warning(f"Grad-CAM hatasi: {e}", exc_info=True)
            return None

    def _async_gradcam_kaydet(self, frame_bgr, img_array, target_path, ch_id):
        try:
            hm = self._make_gradcam_heatmap(img_array)
            if hm is None: 
                log.warning("Grad-CAM: heatmap None (Gradient sifirlandi veya hesaplanamadi).")
                return

            h, w = frame_bgr.shape[:2]
            hm_r = cv2.resize(hm, (w, h), interpolation=cv2.INTER_CUBIC)
            hm_c = cv2.applyColorMap(np.uint8(255 * hm_r), cv2.COLORMAP_JET)

            mask = (hm_r > 0.55).astype(np.float32)
            mask = np.stack([mask]*3, axis=-1)
            
            sup_overlay = cv2.addWeighted(hm_c, 0.70, frame_bgr, 0.30, 0)
            sup = np.where(mask > 0, sup_overlay, frame_bgr).astype(np.uint8)

            cam_path = target_path.replace(".jpg", "_CAM.jpg")
            cv2.imwrite(cam_path, sup)
            log.debug(f"Grad-CAM kaydedildi: {os.path.basename(cam_path)}")
            
            rgb_img = cv2.cvtColor(sup, cv2.COLOR_BGR2RGB)
            
            def _ui():
                try:
                    if not self.is_running: return
                    pil = Image.fromarray(rgb_img)
                    pil.thumbnail((300, 300))
                    itk = ImageTk.PhotoImage(image=pil)
                    
                    lbl = self.lbl_gradcam1 if ch_id == 1 else self.lbl_gradcam2
                    lbl.imgtk_cam = itk 
                    lbl.config(image=itk, text="")
                except Exception as e:
                    log.error(f"UI guncelleme hatasi: {e}")
                    
            self.root.after(0, _ui)
            self.klasor_limitini_koru(os.path.dirname(target_path), self.arsiv_limiti.get())
            
        except Exception as e:
            log.warning(f"Grad-CAM kayit hatasi: {e}")

    # --- ARAYUZ ---
    def setup_ui(self):
        top = tk.Frame(self.root, bg="#eee", pady=5); top.pack(fill="x")
        self.lbl_model = tk.Label(top, text="Model Bekleniyor...", fg="red",
                                  bg="#eee", font=("Arial", 11, "bold"))
        self.lbl_model.pack(side="left", padx=10)
        tk.Button(top, text="Model Sec", command=self.browse_model).pack(side="right", padx=10)
        f_esik = tk.Frame(top, bg="#eee"); f_esik.pack(side="right", padx=10)
        tk.Label(f_esik, text="Guven Esigi:", bg="#eee").pack(side="left")
        tk.Spinbox(f_esik, from_=0.10, to=0.90, increment=0.05,
                   textvariable=self.guven_esigi, width=5, format="%.2f").pack(side="left")
        f_set = tk.Frame(top, bg="#eee"); f_set.pack(side="right", padx=10)
        tk.Label(f_set, text="Arsiv Limiti:", bg="#eee").pack(side="left")
        ttk.Combobox(f_set, textvariable=self.arsiv_limiti,
                     values=[50,100,200], width=5, state="readonly").pack(side="left")

        tabs = ttk.Notebook(self.root)
        self.tab_main  = ttk.Frame(tabs); self.tab_train = ttk.Frame(tabs)
        self.tab_plc   = ttk.Frame(tabs); self.tab_log   = ttk.Frame(tabs)
        tabs.add(self.tab_main,  text=" CIFT KANAL URETIM ")
        tabs.add(self.tab_train, text=" MODEL AYARLARI ")
        tabs.add(self.tab_plc,   text=" PLC AYARLARI ")
        tabs.add(self.tab_log,   text=" LOG ")
        tabs.pack(expand=1, fill="both")
        self.setup_main_tab(); self.setup_train_tab()
        self.setup_plc_tab();  self.setup_log_tab()

    def _gradcam_skala_ciz(self, parent):
        c = tk.Canvas(parent, height=14, bg="#111", highlightthickness=0)
        c.pack(fill="x", padx=4, pady=(0,2))
        renkler = ["#00007f","#0000ff","#007fff","#00ffff",
                   "#7fff7f","#ffff00","#ff7f00","#ff0000","#7f0000"]
        
        def _ciz(event=None):
            w = c.winfo_width()
            if w < 10: return
            c.delete("all")
            seg = w / len(renkler)
            for i, r in enumerate(renkler):
                c.create_rectangle(int(i*seg),0,int((i+1)*seg),14,fill=r,outline="")
            c.create_text(4,  7, text="Dusuk",  fill="#ccc", font=("Arial",6), anchor="w")
            c.create_text(w-4,7, text="Yuksek", fill="#ccc", font=("Arial",6), anchor="e")
            
        c.bind("<Configure>", _ciz)
        c.after(200, _ciz)

    def _gradcam_acikla(self):
        win = tk.Toplevel(self.root)
        win.title("AI Hata Bolgesi - Ne Anlama Geliyor?")
        win.geometry("460x390"); win.resizable(False,False)
        win.configure(bg="#1e1e2e"); win.grab_set()
        tk.Label(win, text="AI Hata Bolgesi Goruntusu", bg="#1e1e2e", fg="#e67e22",
                 font=("Arial",13,"bold")).pack(pady=(20,5))
        frm = tk.Frame(win, bg="#2a2a3e", padx=20, pady=15)
        frm.pack(fill="both", expand=True, padx=15, pady=5)
        for baslik, renk, aciklama in [
            ("KIRMIZI bolge", "#ff4444", "Yapay zekanin EN COK dikkate aldigi alan."),
            ("", "#aaa", "  -> Hata buyuk ihtimalle buradadir."),
            ("SARI bolge",   "#ffcc00", "Orta derecede dikkat edilen alan."),
            ("MAVI bolge",   "#4488ff", "Yapay zekanin pek bakmedigi alan."),
            ("---","",""),
            ("", "#aaa", "Kirmizi KENAR -> Capak / sekil bozuklugu"),
            ("", "#aaa", "Kirmizi YUZEY -> Catlak / dokum hatasi"),
            ("---","",""),
            ("NOT:", "#27ae60", "Bu goruntu SADECE NOK parcalarda olusur."),
        ]:
            if baslik == "---":
                tk.Frame(frm, bg="#444", height=1).pack(fill="x", pady=4); continue
            satir = tk.Frame(frm, bg="#2a2a3e"); satir.pack(fill="x", pady=2)
            if baslik:
                tk.Label(satir, text=baslik, bg="#2a2a3e", fg=renk,
                         font=("Arial",10,"bold"), width=15, anchor="w").pack(side="left")
            tk.Label(satir, text=aciklama, bg="#2a2a3e",
                     fg="white" if baslik else renk,
                     font=("Arial",10), anchor="w").pack(side="left")
        tk.Button(win, text="Anladim", command=win.destroy,
                  bg="#27ae60", fg="white", font=("Arial",11,"bold"),
                  padx=20, pady=6).pack(pady=15)

    def setup_main_tab(self):
        paned = tk.PanedWindow(self.tab_main, orient=tk.HORIZONTAL,
                               bg="#444", sashwidth=10)
        paned.pack(fill="both", expand=True)

        configs = [
            (1, "#2c3e50", "KANAL 1 (SOL)"),
            (2, "#34495e", "KANAL 2 (SAG)"),
        ]
        for ch_id, bg, title in configs:
            frm = tk.Frame(paned, bg=bg); paned.add(frm, width=640)
            tk.Label(frm, text=title, font=("Arial Black",14),
                     bg=bg, fg="white").pack(pady=(8,2))

            img_row = tk.Frame(frm, bg=bg); img_row.pack(fill="both", expand=True, padx=6)

            f_orig = tk.Frame(img_row, bg="#111")
            f_orig.pack(side="left", fill="both", expand=True, padx=(0,2))
            tk.Label(f_orig, text="Orijinal", bg="#111", fg="#aaa", font=("Arial",8)).pack()
            lbl_cam = tk.Label(f_orig, bg="black"); lbl_cam.pack(fill="both", expand=True)

            f_gcam = tk.Frame(img_row, bg="#111")
            f_gcam.pack(side="left", fill="both", expand=True, padx=(2,0))
            gcam_hdr = tk.Frame(f_gcam, bg="#111"); gcam_hdr.pack(fill="x")
            tk.Label(gcam_hdr, text="AI Hata Bolgesi", bg="#111", fg="#e67e22",
                     font=("Arial",8,"bold")).pack(side="left", padx=4)
            tk.Button(gcam_hdr, text="?", font=("Arial",7,"bold"),
                      bg="#555", fg="white", relief="flat", cursor="question_arrow",
                      command=self._gradcam_acikla).pack(side="right", padx=2)
            self._gradcam_skala_ciz(f_gcam)
            lbl_gradcam = tk.Label(f_gcam,
                                   text="NOK tespit edilince\nhata bolgesi\nburada gorulur",
                                   bg="#1a1a1a", fg="#555", font=("Arial",9), justify="center")
            lbl_gradcam.pack(fill="both", expand=True)

            lbl_res = tk.Label(frm, text="---", font=("Arial Black",50), bg=bg, fg="gray")
            lbl_res.pack(pady=(8,2))
            lbl_score = tk.Label(frm, text="Guven: -", bg=bg, fg="#aaa", font=("Arial",10))
            lbl_score.pack()
            lbl_stat = tk.Label(frm, text="OK: 0 | NOK: 0", bg=bg, fg="white", font=("Arial",11))
            lbl_stat.pack(pady=3)
            lbl_metrics = tk.Label(frm, text="Precision: - | Recall: - | F1: -",
                                   bg=bg, fg="#aaa", font=("Arial",9))
            lbl_metrics.pack(pady=(0,6))

            if ch_id == 1:
                self.lbl_cam1=lbl_cam; self.lbl_gradcam1=lbl_gradcam
                self.lbl_res1=lbl_res; self.lbl_score1=lbl_score
                self.lbl_stat1=lbl_stat; self.lbl_metrics1=lbl_metrics
            else:
                self.lbl_cam2=lbl_cam; self.lbl_gradcam2=lbl_gradcam
                self.lbl_res2=lbl_res; self.lbl_score2=lbl_score
                self.lbl_stat2=lbl_stat; self.lbl_metrics2=lbl_metrics

        bot = tk.Frame(self.tab_main, bg="#333", height=60); bot.pack(side="bottom", fill="x")
        self.btn_action = tk.Button(bot, text="SISTEMI BASLAT (KLASOR IZLE)",
                                    command=self.toggle_system, bg="#27ae60", fg="white",
                                    font=("Arial",14,"bold"))
        self.btn_action.pack(side="left", padx=20, pady=10)
        f_plc = tk.Frame(bot, bg="#333"); f_plc.pack(side="right", padx=20)
        tk.Button(f_plc, text="BAGLAN",
                  command=lambda: self.plc_yeniden_baglan(sessiz=False),
                  bg="#3498db", fg="white", font=("Arial",9,"bold")).pack(side="right", padx=5)
        self.lbl_plc = tk.Label(f_plc, text="PLC: ...", bg="#333", fg="yellow", font=("Arial",11))
        self.lbl_plc.pack(side="right", padx=5)

    def setup_log_tab(self):
        frm = tk.Frame(self.tab_log); frm.pack(fill="both", expand=True)
        self.log_text = tk.Text(frm, state="disabled", bg="#1e1e1e", fg="#d4d4d4",
                                font=("Courier",9), wrap="none")
        sb_y = ttk.Scrollbar(frm, orient="vertical",   command=self.log_text.yview)
        sb_x = ttk.Scrollbar(frm, orient="horizontal", command=self.log_text.xview)
        self.log_text.configure(yscrollcommand=sb_y.set, xscrollcommand=sb_x.set)
        sb_y.pack(side="right", fill="y"); sb_x.pack(side="bottom", fill="x")
        self.log_text.pack(fill="both", expand=True)
        tk.Button(frm, text="Temizle", command=self._log_temizle).pack(side="left", padx=5, pady=3)
        tk.Button(frm, text="Kaydet",  command=self._log_kaydet).pack(side="left", padx=5, pady=3)
        self._log_handler_ekle()

    def _log_handler_ekle(self):
        app_ref = self
        class UILogHandler(logging.Handler):
            def __init__(self, widget, app):
                super().__init__(); self.widget=widget; self.app=app
            def emit(self, record):
                if not self.app.is_running: return
                msg = self.format(record) + "\n"
                try:
                    if self.widget.winfo_exists():
                        self.widget.after(0, self._yaz, msg)
                except Exception: pass
            def _yaz(self, msg):
                try:
                    if not self.widget.winfo_exists(): return
                    self.widget.configure(state="normal")
                    self.widget.insert("end", msg)
                    self.widget.see("end")
                    self.widget.configure(state="disabled")
                except Exception: pass

        logger = logging.getLogger("MKE_AI")
        logger.handlers = [h for h in logger.handlers if not isinstance(h, UILogHandler)]
        handler = UILogHandler(self.log_text, app_ref)
        handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s",
                                               datefmt="%H:%M:%S"))
        logger.addHandler(handler); self._ui_log_handler = handler

    def _log_temizle(self):
        self.log_text.configure(state="normal"); self.log_text.delete("1.0","end")
        self.log_text.configure(state="disabled")

    def _log_kaydet(self):
        p = filedialog.asksaveasfilename(defaultextension=".log", filetypes=[("Log","*.log")])
        if p:
            with open(p,"w",encoding="utf-8") as f: f.write(self.log_text.get("1.0","end"))
            messagebox.showinfo("Kaydedildi", f"Log kaydedildi:\n{p}")

    # --- PLC AYARLARI ---
    def setup_plc_tab(self):
        outer = tk.Frame(self.tab_plc, bg="#f0f0f0")
        outer.pack(fill="both", expand=True, padx=40, pady=30)
        tk.Label(outer, text="PLC Baglanti Ayarlari", font=("Arial",15,"bold"),
                 bg="#f0f0f0").grid(row=0, column=0, columnspan=3, sticky="w", pady=(0,5))
        tk.Label(outer, text="S7-1200: Rack=0 Slot=1  |  S7-300/400: Rack=0 Slot=2",
                 bg="#f0f0f0", fg="#555", font=("Arial",9)).grid(
                     row=1, column=0, columnspan=3, sticky="w", pady=(0,15))

        pf = tk.LabelFrame(outer, text="Hizli On Ayar", bg="#f0f0f0",
                            font=("Arial",10,"bold"), padx=10, pady=8)
        pf.grid(row=2, column=0, columnspan=3, sticky="ew", pady=(0,15))
        tk.Button(pf, text="S7-1200 / S7-1500",
                  command=lambda: (self.plc_rack.set(0), self.plc_slot.set(1)),
                  bg="#2980b9", fg="white", font=("Arial",10,"bold"), width=20).pack(side="left", padx=(0,10))
        tk.Button(pf, text="S7-300 / S7-400",
                  command=lambda: (self.plc_rack.set(0), self.plc_slot.set(2)),
                  bg="#7f8c8d", fg="white", font=("Arial",10,"bold"), width=20).pack(side="left")

        cf = tk.LabelFrame(outer, text="Baglanti Parametreleri", bg="#f0f0f0",
                            font=("Arial",10,"bold"), padx=15, pady=10)
        cf.grid(row=3, column=0, columnspan=3, sticky="ew", pady=(0,15))
        for i,(lbl,var,hint) in enumerate([
            ("PLC IP Adresi:", self.plc_ip, "Orn: 192.168.0.1"),
            ("Rack:",          self.plc_rack, "S7-1200: 0"),
            ("Slot:",          self.plc_slot, "S7-1200: 1  |  S7-300: 2"),
            ("Data Block:",    self.plc_db,   "TIA Portal DB numarasi"),
        ]):
            tk.Label(cf, text=lbl, bg="#f0f0f0", anchor="e", width=18).grid(row=i,column=0,sticky="e",padx=5,pady=4)
            if i == 0:
                tk.Entry(cf, textvariable=var, width=18, font=("Courier",10)).grid(row=i,column=1,sticky="w",padx=5)
            else:
                tk.Spinbox(cf, from_=0, to=65535, textvariable=var, width=8).grid(row=i,column=1,sticky="w",padx=5)
            tk.Label(cf, text=hint, bg="#f0f0f0", fg="#888", font=("Arial",8)).grid(row=i,column=2,sticky="w")

        of = tk.LabelFrame(outer, text="DB Offset Ayarlari (INT = 2 byte)", bg="#f0f0f0",
                            font=("Arial",10,"bold"), padx=15, pady=10)
        of.grid(row=4, column=0, columnspan=3, sticky="ew", pady=(0,15))
        for i,(lbl,var) in enumerate([("Kanal 1 Offset:",self.plc_offset_ch1),
                                       ("Kanal 2 Offset:",self.plc_offset_ch2)]):
            tk.Label(of, text=lbl, bg="#f0f0f0", anchor="e", width=18).grid(row=i,column=0,sticky="e",padx=5,pady=4)
            tk.Spinbox(of, from_=0, to=9998, increment=2, textvariable=var, width=8).grid(row=i,column=1,sticky="w",padx=5)
            tk.Label(of, text="INT (2 byte) -- cift sayi", bg="#f0f0f0", fg="#888",
                     font=("Arial",8)).grid(row=i,column=2,sticky="w")

        bf = tk.Frame(outer, bg="#f0f0f0"); bf.grid(row=5, column=0, columnspan=3, sticky="w", pady=10)
        tk.Button(bf, text="Ayarlari Kaydet & Baglan",
                  command=lambda: self.plc_yeniden_baglan(sessiz=False),
                  bg="#27ae60", fg="white", font=("Arial",12,"bold"),
                  padx=15, pady=6).pack(side="left", padx=(0,15))
        tk.Button(bf, text="Baglantiyi Kes", command=self._plc_baglantiyi_kes,
                  bg="#c0392b", fg="white", font=("Arial",12,"bold"),
                  padx=15, pady=6).pack(side="left")
        self.lbl_plc_tab = tk.Label(outer, text="Baglanti durumu bekleniyor...",
                                    bg="#f0f0f0", fg="#555", font=("Arial",10))
        self.lbl_plc_tab.grid(row=6, column=0, columnspan=3, sticky="w", pady=5)

    def _plc_baglantiyi_kes(self):
        try: self.plc.client.disconnect()
        except Exception: pass
        self.plc.connected = False; self.plc.simulasyon_modu = True
        self.lbl_plc.config(text="PLC YOK (Simulasyon)", fg="red")
        self.lbl_plc_tab.config(text="Baglanti kesildi -- Simulasyon modu.", fg="orange")
        log.info("PLC baglantisi manuel kesildi.")

    def plc_ayarlari_uygula(self):
        self.plc.ip=self.plc_ip.get().strip(); self.plc.rack=self.plc_rack.get()
        self.plc.slot=self.plc_slot.get(); self.plc.db_num=self.plc_db.get()
        self.plc.connected=False; self.plc.simulasyon_modu=False
        log.info(f"PLC ayarlari: IP={self.plc.ip} Rack={self.plc.rack} Slot={self.plc.slot} DB={self.plc.db_num}")

    def plc_yeniden_baglan(self, sessiz=False):
        self.plc_ayarlari_uygula()
        durum = self.plc.baglan(); ip_str = self.plc.ip
        if durum:
            msg = f"PLC BAGLI ({ip_str}) Rack={self.plc.rack} Slot={self.plc.slot} DB={self.plc.db_num}"
            self.lbl_plc.config(text=f"PLC BAGLI ({ip_str})", fg="green")
            self.lbl_plc_tab.config(text=msg, fg="green")
            if not sessiz: messagebox.showinfo("Basarili", f"PLC Baglantisi Kuruldu!\nIP: {ip_str}")
        else:
            err = f"Baglanti basarisiz -- IP: {ip_str} Rack={self.plc.rack} Slot={self.plc.slot}"
            self.lbl_plc.config(text="PLC YOK (Simulasyon)", fg="red")
            self.lbl_plc_tab.config(text=err, fg="red")
            if not sessiz:
                messagebox.showerror("Hata", f"PLC'ye Ulasilamadi!\nIP: {ip_str}\n"
                                     f"Rack={self.plc.rack}  Slot={self.plc.slot}")

    # --- EGITIM ---
    def setup_train_tab(self):
        outer = tk.Frame(self.tab_train, bg="#f5f5f5")
        outer.pack(fill="both", expand=True)

        sol = tk.Frame(outer, bg="#f5f5f5", padx=30, pady=25)
        sol.pack(side="left", fill="y")

        tk.Label(sol, text="Model Ayarlari", font=("Arial",14,"bold"),
                 bg="#f5f5f5").grid(row=0, column=0, columnspan=3, sticky="w", pady=(0,15))

        hp = [
            ("Epoch Sayisi:", self.epoch_sayisi, 1, 50, 1,
             "Tum verinin kac kez gorulecegi.\nDaha fazla -> iyi ogrenme\nancak overfitting riski."),
            ("Batch Boyutu:", self.batch_size, 1, 32, 1,
             "Her adimda islenen goruntu sayisi.\nKucuk (4-8): az RAM, gozlu\nBuyuk (16-32): kararli, cok RAM"),
            ("Learning Rate:", self.learning_rate, 1e-7, 1e-2, 1e-6,
             "Agirliklarin adim buyuklugu.\nCok buyuk->kararsiz.\nInce ayar icin 1e-5 idealdir."),
        ]
        for i, (lbl, var, lo, hi, inc, acik) in enumerate(hp, start=1):
            tk.Label(sol, text=lbl, bg="#f5f5f5", anchor="e",
                     font=("Arial",10)).grid(row=i, column=0, sticky="e", padx=5, pady=6)
            if isinstance(var.get(), float):
                w = tk.Spinbox(sol, from_=lo, to=hi, increment=inc, textvariable=var,
                               width=12, format="%.7f")
            else:
                w = tk.Spinbox(sol, from_=lo, to=hi, increment=inc, textvariable=var, width=8)
            w.grid(row=i, column=1, sticky="w", padx=5, pady=6)
            tk.Label(sol, text="(?)", fg="#3498db", bg="#f5f5f5",
                     font=("Arial",9,"bold"), cursor="question_arrow").grid(
                         row=i, column=2, padx=5)
            tk.Label(sol, text=acik, bg="#eaf0fb", fg="#333",
                     font=("Arial",8), justify="left", relief="flat",
                     padx=6, pady=4, wraplength=230).grid(
                         row=i+10, column=0, columnspan=3, sticky="w",
                         padx=5, pady=(0,4))

        sep = tk.Frame(sol, bg="#ddd", height=1)
        sep.grid(row=4, column=0, columnspan=3, sticky="ew", pady=15)

        tk.Label(sol, text="Egitim Klasoru:", bg="#f5f5f5",
                 font=("Arial",10,"bold")).grid(row=5, column=0, sticky="e", padx=5)
        self.egitim_klasoru = tk.StringVar(value="Klasor secilmedi")
        lbl_klasor = tk.Label(sol, textvariable=self.egitim_klasoru,
                              bg="white", fg="#555", font=("Courier",9),
                              relief="sunken", width=28, anchor="w", padx=4)
        lbl_klasor.grid(row=5, column=1, columnspan=2, sticky="ew", padx=5, pady=4)

        tk.Button(sol, text="  Klasor Sec...",
                  command=self._egitim_klasoru_sec,
                  bg="#2980b9", fg="white", font=("Arial",10,"bold"),
                  padx=10, pady=5).grid(row=6, column=0, columnspan=3,
                                        sticky="w", padx=5, pady=(4,2))

        self.btn_egitim_baslat = tk.Button(
            sol, text="  Egitimi Baslat",
            command=self._egitim_baslat,
            bg="#8e44ad", fg="white", font=("Arial",11,"bold"),
            padx=10, pady=6, state="disabled")
        self.btn_egitim_baslat.grid(row=7, column=0, columnspan=3,
                                     sticky="w", padx=5, pady=(2,10))
        tk.Button(sol, text="  Modeli Sifirdan Kur ",
                  command=self._modeli_sifirdan_kur,
                  bg="#e67e22", fg="white", font=("Arial",10,"bold"),
                  padx=10, pady=5).grid(row=8, column=0, columnspan=3,
                                        sticky="w", padx=5, pady=(2,10))
        tk.Button(sol, text="  Holdout Testi Başlat ",
                  command=self.holdout_test_baslat,
                  bg="#16a085", fg="white", font=("Arial",10,"bold"),
                  padx=10, pady=5).grid(row=9, column=0, columnspan=3,
                                        sticky="w", padx=5, pady=(2,10))

        orta = tk.Frame(outer, bg="#fffdf5", relief="solid", bd=1)
        orta.pack(side="left", fill="y", padx=(0,10), pady=20, ipadx=10, ipady=10)

        tk.Label(orta, text="Klasor Yapisi Rehberi",
                 font=("Arial", 11, "bold"), bg="#fffdf5", fg="#c0392b").pack(
                     anchor="w", padx=12, pady=(10,6))

        aciklama = (
            "Egitim klasoru icinde tam olarak\n"
            "2 alt klasor olmalidir:\n"
            "  OK   -> temiz (hatasiz) parcalar\n"
            "  NOK  -> hatali parcalar\n\n"
            "Klasor isimleri tam olarak bu\n"
            "sekilde olmalidir (buyuk harf,\n"
            "kucuk harf farketmez)."
        )
        tk.Label(orta, text=aciklama, font=("Arial", 9), bg="#fffdf5",
                 fg="#555", justify="left").pack(anchor="w", padx=12, pady=(0,8))

        c = tk.Canvas(orta, width=220, height=200, bg="#fffdf5",
                      highlightthickness=0)
        c.pack(padx=12, pady=(0,6))

        def ciz_agac():
            c.delete("all")
            KLASOR_BG  = "#f0c060"
            KLASOR_FG  = "#7a5000"
            OK_BG      = "#d5f5e3"
            OK_FG      = "#1e8449"
            NOK_BG     = "#fadbd8"
            NOK_FG     = "#922b21"
            DOSYA_BG   = "#eaf4fb"
            DOSYA_FG   = "#1a5276"
            LINE_COL   = "#bbb"

            def kutu(x, y, w, h, bg, fg, yazi, kucuk=False):
                c.create_rectangle(x, y, x+w, y+h, fill=bg, outline=fg, width=2)
                font = ("Arial", 8) if kucuk else ("Arial", 9, "bold")
                c.create_text(x+w//2, y+h//2, text=yazi, fill=fg, font=font)

            def cizgi(x1, y1, x2, y2):
                c.create_line(x1, y1, x2, y2, fill=LINE_COL, width=1, dash=(4,2))

            kutu(60, 8, 100, 28, KLASOR_BG, KLASOR_FG, "egitim_klasoru")
            cizgi(110, 36, 110, 52)
            cizgi(55, 52, 165, 52)
            cizgi(55, 52, 55, 68)
            cizgi(165, 52, 165, 68)

            kutu(15, 68, 78, 28, OK_BG, OK_FG, "OK  (veya ok)")
            kutu(107, 68, 90, 28, NOK_BG, NOK_FG, "NOK  (veya nok)")

            cizgi(54, 96, 54, 112)
            for i, isim in enumerate(["parca_001.jpg", "parca_002.jpg", "..."]):
                y_ = 112 + i * 22
                cizgi(54, y_, 66, y_)
                kutu(66, y_-9, 108, 18, DOSYA_BG, DOSYA_FG, isim, kucuk=True)

            cizgi(152, 96, 152, 112)
            for i, isim in enumerate(["hata_001.jpg", "hata_002.jpg", "..."]):
                y_ = 112 + i * 22
                cizgi(152, y_, 164, y_)
                kutu(164, y_-9, 108, 18, NOK_BG, NOK_FG, isim, kucuk=True)

        c.after(100, ciz_agac)
        c.bind("<Configure>", lambda e: ciz_agac())

        not_frm = tk.Frame(orta, bg="#fef9e7", relief="solid", bd=1)
        not_frm.pack(fill="x", padx=12, pady=(0,10))
        tk.Label(not_frm,
                 text="NOT: Her klasorde en az\n10-20 goruntu olmali.\nDaha fazla goruntu =\ndaha iyi egitim.",
                 font=("Arial", 8), bg="#fef9e7", fg="#7d6608",
                 justify="left", padx=6, pady=5).pack()

        sag = tk.Frame(outer, bg="white", relief="sunken", bd=1)
        sag.pack(side="left", fill="both", expand=True, padx=(0,20), pady=20)

        tk.Label(sag, text="Egitim Durumu & Sonuclari",
                 font=("Arial",12,"bold"), bg="white").pack(anchor="w", padx=15, pady=(12,5))

        self.lbl_train_durum = tk.Label(sag, text="Bekliyor...",
                                        font=("Arial",11), bg="white", fg="#888")
        self.lbl_train_durum.pack(anchor="w", padx=15, pady=(0,5))

        self.progress = ttk.Progressbar(sag, length=380, mode="determinate")
        self.progress.pack(padx=15, pady=(0,10), fill="x")

        self.lbl_train_msg = tk.Label(sag, text="", font=("Courier",9),
                                      bg="white", fg="#333", justify="left")
        self.lbl_train_msg.pack(anchor="w", padx=15)

        sep2 = tk.Frame(sag, bg="#ddd", height=1)
        sep2.pack(fill="x", padx=15, pady=10)

        tk.Label(sag, text="Epoch Gecmisi:", font=("Arial",10,"bold"),
                 bg="white").pack(anchor="w", padx=15, pady=(0,4))

        tablo_frm = tk.Frame(sag, bg="white")
        tablo_frm.pack(fill="both", expand=True, padx=15, pady=(0,10))

        self.epoch_text = tk.Text(tablo_frm, height=12, font=("Courier",9),
                                  bg="#fafafa", fg="#333", state="disabled",
                                  relief="flat", wrap="none")
        sb = ttk.Scrollbar(tablo_frm, command=self.epoch_text.yview)
        self.epoch_text.configure(yscrollcommand=sb.set)
        sb.pack(side="right", fill="y")
        self.epoch_text.pack(fill="both", expand=True)

        self.epoch_text.tag_configure("baslik", foreground="#2980b9", font=("Courier",9,"bold"))
        self.epoch_text.tag_configure("iyi",    foreground="#27ae60")
        self.epoch_text.tag_configure("normal", foreground="#333")
        self.epoch_text.tag_configure("bitti",  foreground="#8e44ad", font=("Courier",9,"bold"))

        self._egitim_devam_ediyor = False

    def _egitim_klasoru_sec(self):
        d = filedialog.askdirectory(title="OK ve NOK alt-klasorlerini iceren klasoru secin")
        if d:
            self.egitim_klasoru.set(d)
            self.btn_egitim_baslat.config(state="normal")
            log.info(f"Egitim klasoru secildi: {d}")

    def _egitim_baslat(self):
        if self._egitim_devam_ediyor:
            messagebox.showwarning("Uyari", "Egitim zaten devam ediyor!")
            return
        d = self.egitim_klasoru.get()
        if d == "Klasor secilmedi" or not os.path.isdir(d):
            messagebox.showerror("Hata", "Lutfen once gecerli bir egitim klasoru secin.")
            return
        ep = self.epoch_sayisi.get() 
        bs = self.batch_size.get()
        lr = self.learning_rate.get()
        arsiv_lim = self.arsiv_limiti.get()
        guven_esigi = self.guven_esigi.get()
        threading.Thread(target=self.run_train, args=(d, ep, bs, lr, arsiv_lim, guven_esigi), daemon=True).start()
    def _epoch_text_yaz(self, mesaj, tag="normal"):
        def _yaz():
            try:
                self.epoch_text.configure(state="normal")
                self.epoch_text.insert("end", mesaj + "\n", tag)
                self.epoch_text.see("end")
                self.epoch_text.configure(state="disabled")
            except Exception: pass
        self.root.after(0, _yaz)

    def start_train(self):
        pass  

    def run_train(self, d, ep, bs, lr, arsiv_lim, guven_esigi):
        if not tf:
            self.root.after(0, lambda: messagebox.showerror("Hata","TensorFlow yuklenmedi.")); return
        if not self.model:
            self.root.after(0, lambda: messagebox.showerror("Hata","Once model yukleyiniz.")); return

        self._egitim_devam_ediyor = True
        
        def _ui_baslat():
            try:
                self.btn_egitim_baslat.config(state="disabled", text="  Egitim Devam Ediyor...")
                self.progress.config(value=0)
                self.lbl_train_durum.config(text="Egitim baslatiliyor...", fg="#e67e22")
                self.lbl_train_msg.config(text="")
                self.epoch_text.configure(state="normal")
                self.epoch_text.delete("1.0", "end")
                self.epoch_text.configure(state="disabled")
            except Exception: pass
        self.root.after(0, _ui_baslat)

        baslik = (f"{'Epoch':>6}  {'Loss':>8}  {'Acc':>8}  "
                  f"{'Val-Loss':>10}  {'Val-Acc':>9}  Durum")
        self._epoch_text_yaz(baslik, "baslik")
        self._epoch_text_yaz("-" * 62, "baslik")

        log.info(f"Egitim baslatildi: {d} | epoch={ep} bs={bs} lr={lr:.2e}")

        self._ani_chars = ["", ".", "..", "..."]
        self._ani_idx   = 0

        def _ani_guncelle():
            if not self._egitim_devam_ediyor: return
            self._ani_idx = (self._ani_idx + 1) % len(self._ani_chars)
            try:
                mevcut = self.lbl_train_durum.cget("text").split("   ")[0]
                self.lbl_train_durum.config(
                    text=f"{mevcut}   {self._ani_chars[self._ani_idx]}")
                self.root.after(400, _ani_guncelle)
            except Exception: pass
        self.root.after(400, _ani_guncelle)

        class CB(tf.keras.callbacks.Callback):
            def __init__(self_, app, total):
                self_.app = app; self_.total = total; self_.gecmis = []

            def on_train_begin(self_, logs=None):
                self_.app.root.after(0, lambda: self_.app.lbl_train_durum.config(
                    text=f"Egitim devam ediyor  (0/{self_.total} epoch)", fg="#e67e22"))

            def on_epoch_end(self_, epoch, logs=None):
                logs = logs or {}
                loss     = logs.get('loss',      0)
                acc      = logs.get('accuracy',  0)
                val_loss = logs.get('val_loss',  0)
                val_acc  = logs.get('val_accuracy', logs.get('val_acc', 0))
                pct      = ((epoch+1) / self_.total) * 100

                durum_ikon = ""
                if self_.gecmis:
                    if loss < self_.gecmis[-1]['loss']:
                        durum_ikon = "v"   
                    elif loss > self_.gecmis[-1]['loss']:
                        durum_ikon = "^"   
                self_.gecmis.append({'loss': loss, 'acc': acc})

                satirr = (f"{epoch+1:>6}  {loss:>8.4f}  {acc*100:>7.2f}%  "
                          f"{val_loss:>10.4f}  {val_acc*100:>8.2f}%   {durum_ikon}")
                tag = "iyi" if durum_ikon == "v" else "normal"

                msg_ui = (f"Epoch {epoch+1}/{self_.total} | "
                          f"loss={loss:.4f} | acc={acc*100:.1f}% | "
                          f"val_loss={val_loss:.4f} | val_acc={val_acc*100:.1f}%")
                log.info(msg_ui)

                def _guncelle(s=satirr, t=tag, p=pct, m=msg_ui, e=epoch):
                    try:
                        self_.app.progress.config(value=p)
                        self_.app.lbl_train_durum.config(
                            text=f"Egitim devam ediyor  ({e+1}/{self_.total} epoch)", fg="#e67e22")
                        self_.app.lbl_train_msg.config(text=m)
                        self_.app._epoch_text_yaz(s, t)
                    except Exception: pass
                self_.app.root.after(0, _guncelle)

        try:
            
            d_train = os.path.join(d, "train")
            d_test  = os.path.join(d, "test")

            train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
                rescale=1./255,
                rotation_range=360,       
                width_shift_range=0.05,   
                height_shift_range=0.05,  
                zoom_range=0.05,          
                horizontal_flip=True,    
                vertical_flip =True,
                brightness_range = [0.85, 1.15],
                fill_mode='nearest'      
                )
            
            test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)
            
            tg = train_datagen.flow_from_directory(d_train, target_size=IMG_SIZE, batch_size=bs, class_mode='binary')
            vg = test_datagen.flow_from_directory(d_test,  target_size=IMG_SIZE, batch_size=bs, class_mode='binary')
            if tg.samples == 0: raise ValueError("Egitim klasorunde goruntu bulunamadi.")

            tr_list = tg.classes.tolist()
            val_list = vg.classes.tolist()
            idx_ok = tg.class_indices.get('OK', tg.class_indices.get('ok', 1))
            idx_nok = tg.class_indices.get('NOK', tg.class_indices.get('nok', 0))

            n_ok  = tr_list.count(idx_ok)
            n_nok = tr_list.count(idx_nok)
            toplam = n_ok + n_nok
            class_weight = {
            idx_ok:  toplam / (2.0 * n_ok)  if n_ok  > 0 else 1.0,
            idx_nok: toplam / (2.0 * n_nok) if n_nok > 0 else 1.0
            }
            log.info(f"Class weight -> OK: {class_weight[idx_ok]:.2f} | NOK: {class_weight[idx_nok]:.2f}")
            self._epoch_text_yaz(f"Veri Ozeti: Toplam {tg.samples + vg.samples} goruntu", "baslik")
            self._epoch_text_yaz(f" -> Egitim: {tg.samples} (OK: {tr_list.count(idx_ok)} | NOK: {tr_list.count(idx_nok)})", "normal")
            self._epoch_text_yaz(f" -> Test  : {vg.samples} (OK: {val_list.count(idx_ok)} | NOK: {val_list.count(idx_nok)})", "normal")
            self._epoch_text_yaz("-" * 62, "baslik")

            with self.model_lock:
                self.model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                                   loss='binary_crossentropy',
                                   metrics=[
                                       'accuracy',
                                       tf.keras.metrics.Precision(name='precision'),
                                       tf.keras.metrics.Recall(name='recall'),
                                       tf.keras.metrics.AUC(name='auc')
                                           ])
                    
                gecmis = self.model.fit(
                    tg, epochs=ep, validation_data=vg,  
                    class_weight=class_weight,
                    callbacks=[CB(self, ep)], verbose=0)

                if any(isinstance(l, tf.keras.Model) and 'mobilenet' in l.name.lower() 
                       for l in self.model.layers):
                    sp = os.path.join(os.getcwd(), DEFAULT_MODEL_NAME_MOBILE)
                else:                
                    sp = os.path.join(os.getcwd(), DEFAULT_MODEL_NAME)
                self.model.save(sp)
                log.info(f"Model kaydedildi: {sp}")

            son_loss    = gecmis.history['loss'][-1]
            son_acc     = gecmis.history['accuracy'][-1]
            son_vl      = gecmis.history.get('val_loss',  [0])[-1]
            son_va      = gecmis.history.get('val_accuracy', gecmis.history.get('val_acc',[0]))[-1]
            
            son_prec    = gecmis.history.get('precision', [0])[-1]
            son_rec     = gecmis.history.get('recall', [0])[-1]
            son_auc     = gecmis.history.get('auc', [0])[-1]
            f1_score    = (2 * son_prec * son_rec) / (son_prec + son_rec) if (son_prec + son_rec) > 0 else 0

            def _bitti():
                try:
                    self._egitim_devam_ediyor = False
                    self.btn_egitim_baslat.config(state="normal", text="  Egitimi Baslat")
                    self.progress.config(value=100)
                    self.lbl_train_durum.config(text="Egitim tamamlandi!", fg="#27ae60")
                    self._epoch_text_yaz("", "normal")
                    self._epoch_text_yaz("=" * 62, "bitti")
                    self._epoch_text_yaz("EGITIM TAMAMLANDI", "bitti")
                    self._epoch_text_yaz("-" * 62, "normal")
                    self._epoch_text_yaz("DETAYLI PERFORMANS TABLOSU:", "baslik")
                    self._epoch_text_yaz(f"Dogruluk (Accuracy)  : %{son_acc*100:.1f}", "normal")
                    self._epoch_text_yaz(f"Kesinlik (Precision) : %{son_prec*100:.1f}", "normal")
                    self._epoch_text_yaz(f"Duyarlilik (Recall)  : %{son_rec*100:.1f}", "normal")
                    self._epoch_text_yaz(f"F1 Skoru             : %{f1_score*100:.1f}", "normal")
                    self._epoch_text_yaz(f"AUC                  : {son_auc:.3f}", "normal")
                    self._epoch_text_yaz(f"Yanlis Negatif (FN)  : %{(1-son_rec)*100:.1f}", "normal")
                    self._epoch_text_yaz("-" * 62, "normal")
                    self._epoch_text_yaz("MODEL KATMAN VE PARAMETRE OZETI:", "baslik")
                    self.model.summary(print_fn=lambda x: self._epoch_text_yaz(x, "normal"))
                    self._epoch_text_yaz("-" * 62, "normal")
                    self._epoch_text_yaz(
                        f"Son egitim    : loss={son_loss:.4f}  acc={son_acc*100:.2f}%", "bitti")
                    self._epoch_text_yaz(
                        f"Son dogrulama : loss={son_vl:.4f}  acc={son_va*100:.2f}%", "bitti")
                    self._epoch_text_yaz(
                        f"Model kaydedildi: {DEFAULT_MODEL_NAME}", "bitti")
                    self._epoch_text_yaz("=" * 62, "bitti")
                    self.lbl_train_msg.config(
                        text=f"Son: loss={son_loss:.4f} acc={son_acc*100:.1f}% | "
                             f"val_loss={son_vl:.4f} val_acc={son_va*100:.1f}%")
                except Exception: pass
            self.root.after(0, _bitti)

        except Exception as e:
            log.error(f"Egitim hatasi: {e}")
            def _hata(err=str(e)):
                try:
                    self._egitim_devam_ediyor = False
                    self.btn_egitim_baslat.config(state="normal", text="  Egitimi Baslat")
                    self.lbl_train_durum.config(text=f"HATA: {err}", fg="#e74c3c")
                    self._epoch_text_yaz(f"HATA: {err}", "baslik")
                    messagebox.showerror("Egitim Hatasi", err)
                except Exception: pass
            self.root.after(0, _hata)

    # --- SISTEM ---
    def toggle_system(self):
        if self.btn_action['text'].startswith("SISTEMI BASLAT"):
            ev1=DosyaYakalayici(self,1); self.observer1=Observer()
            self.observer1.schedule(ev1,self.dir_ch1_in,recursive=False); self.observer1.start()
            ev2=DosyaYakalayici(self,2); self.observer2=Observer()
            self.observer2.schedule(ev2,self.dir_ch2_in,recursive=False); self.observer2.start()
            self.btn_action.config(text="SISTEMI DURDUR",bg="#c0392b")
            self.lbl_res1.config(text="HAZIR",fg="white")
            self.lbl_res2.config(text="HAZIR",fg="white")
            log.info("Sistem baslatildi.")
        else:
            if self.observer1: self.observer1.stop(); self.observer1=None
            if self.observer2: self.observer2.stop(); self.observer2=None
            self.btn_action.config(text="SISTEMI BASLAT (KLASOR IZLE)",bg="#27ae60")
            self.lbl_res1.config(text="---",fg="gray"); self.lbl_res2.config(text="---",fg="gray")
            log.info("Sistem durduruldu.")
    

    def dosya_analiz(self, path, ch_id):
        if not self.is_running: return
        img = None
        for deneme in range(20):
            if not self.is_running: return
            try:
                arr=np.fromfile(path,np.uint8)
                if arr.size>0:
                    img=cv2.imdecode(arr,cv2.IMREAD_COLOR)
                    if img is not None: break
            except PermissionError:
                if deneme<19: time.sleep(0.020); continue
                log.warning(f"Izin alinamadi (CH{ch_id}): {path}")
            except Exception as e:
                log.warning(f"Okuma hatasi (CH{ch_id}): {e}")
            time.sleep(0.020)
        if img is not None: self.process_image(img, path, ch_id)
        elif not os.path.exists(path): log.debug(f"Tasinmis, atlandi (CH{ch_id})")
        else: log.error(f"Goruntu okunamadi (CH{ch_id}): {path}")

    def _test_cakisma(self, event=None):
        log.info("=== [Stres Testi] CAKISMA TESTI BASLADI ===")
        
        ch1_offset_degeri = self.plc_offset_ch1.get()
        ch2_offset_degeri = self.plc_offset_ch2.get()

        ch1_lock = threading.Lock()
        ch2_lock = threading.Lock()

        def _arka_plan_testi():
            hatalar = []
            istatistik = {'CH1': {'OK': 0, 'NOK': 0}, 'CH2': {'OK': 0, 'NOK': 0}}
            istatistik_lock = threading.Lock() 
            
            def uretim_ch1(sinyal_degeri):
                with ch1_lock: 
                    try:         
                        basarili = self.plc.yaz_ve_onayla(sinyal_degeri, ch1_offset_degeri, 12, 0)
                        
                        if basarili:
                            with istatistik_lock:
                                if sinyal_degeri == 1: istatistik['CH1']['OK'] += 1
                                else: istatistik['CH1']['NOK'] += 1
                        else:
                            hatalar.append("CH1: Handshake Timeout")
                    except Exception as e: 
                        hatalar.append(f"CH1: {e}")
                
            def uretim_ch2(sinyal_degeri):
                with ch2_lock: 
                    try:                         
                        basarili = self.plc.yaz_ve_onayla(sinyal_degeri, ch2_offset_degeri, 12, 1)
                        
                        if basarili:
                            with istatistik_lock:
                                if sinyal_degeri == 1: istatistik['CH2']['OK'] += 1
                                else: istatistik['CH2']['NOK'] += 1
                        else:
                            hatalar.append("CH2: Handshake Timeout")
                    except Exception as e: 
                        hatalar.append(f"CH2: {e}")

            threads = []
            test_t0 = time.perf_counter()
            
            for i in range(50):
                ch1_rastgele = random.choice([1, 2])
                ch2_rastgele = random.choice([1, 2])
                
                t1 = threading.Thread(target=uretim_ch1, args=(ch1_rastgele,))
                t2 = threading.Thread(target=uretim_ch2, args=(ch2_rastgele,))
                threads.extend([t1, t2])
                t1.start()
                t2.start()
                
            for t in threads:
                t.join()
                
            test_t1 = time.perf_counter()
            toplam_sure = test_t1 - test_t0
                
            if not hatalar:
                log.info(f"[Stres Testi] BASARILI! Sinyaller PLC tarafindan teyit edildi.")
                log.info(f"[Stres Testi] 100 paketin tam senkronize islenmesi {toplam_sure:.2f} saniye surdu.")
                log.info(f"[Stres Testi] DAGILIM -> CH1: {istatistik['CH1']['OK']} OK, {istatistik['CH1']['NOK']} NOK | CH2: {istatistik['CH2']['OK']} OK, {istatistik['CH2']['NOK']} NOK")
            else:
                log.error(f"[Stres Testi] YAZILIM/PLC CAKISMA HATASI: {len(hatalar)} adet hata olustu!")

        threading.Thread(target=_arka_plan_testi, daemon=True).start()
        
   

    def process_image(self, frame_bgr, file_path, ch_id):
        e2e_t0 = time.perf_counter()
        if not self.is_running or not tf or not self.model: return
        try:
            f_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            im_pil = Image.fromarray(f_rgb); im_pil.thumbnail((300, 300))
            itk = ImageTk.PhotoImage(image=im_pil)
    
            lbl_cam  = self.lbl_cam1  if ch_id == 1 else self.lbl_cam2
            lbl_res  = self.lbl_res1  if ch_id == 1 else self.lbl_res2
            lbl_sc   = self.lbl_score1 if ch_id == 1 else self.lbl_score2
            lbl_stat = self.lbl_stat1 if ch_id == 1 else self.lbl_stat2
            lbl_mtr  = self.lbl_metrics1 if ch_id == 1 else self.lbl_metrics2
            lbl_grad = self.lbl_gradcam1 if ch_id == 1 else self.lbl_gradcam2
    
            lbl_cam.imgtk = itk; lbl_cam.config(image=itk)
    
            frame_display_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    
            with self.model_lock:
                inp   = cv2.resize(frame_rgb, IMG_SIZE).astype("float32") / 255.0
                inp   = np.expand_dims(inp, axis=0)
                score = float(self.model(inp, training=False)[0][0])
    
            esik = self.guven_esigi.get(); is_faulty = score < esik
    
            if is_faulty:
                lbl_res.config(text="NOK", fg="#e74c3c")
                self.stats[ch_id]['nok'] += 1; plc_val = 2
                log.info(f"CH{ch_id} -> NOK | score={score:.3f} | {os.path.basename(file_path)}")
            else:
                lbl_res.config(text="OK", fg="#2ecc71")
                self.stats[ch_id]['ok'] += 1; plc_val = 1
                log.info(f"CH{ch_id} -> OK  | score={score:.3f} | {os.path.basename(file_path)}")
                lbl_grad.config(image="", text="NOK tespit edilince\nhata bolgesi\nburada gorulur")
    
            lbl_sc.config(text=f"Guven: {score:.3f}  (esik: {esik:.2f})")
            offset = self.plc_offset_ch1.get() if ch_id == 1 else self.plc_offset_ch2.get()
            self.plc.yaz_ve_onayla(
                karar_degeri    = plc_val,
                karar_offset    = offset,
                ack_byte_offset = 12,
                ack_bit_index   = 0 if ch_id == 1 else 1
            )
            e2e_sure = (time.perf_counter() - e2e_t0) * 1000
            self.e2e_gecmisi[ch_id].append(e2e_sure)
            ortalama_e2e = sum(self.e2e_gecmisi[ch_id]) / len(self.e2e_gecmisi[ch_id])
            log.info(f"[E2E Test] CH{ch_id} E2E Gecikme: {e2e_sure:.2f} ms | GUNCEL ORTALAMA: {ortalama_e2e:.2f} ms")
    
            ok_ = self.stats[ch_id]['ok']; nok_ = self.stats[ch_id]['nok']
            lbl_stat.config(text=f"OK: {ok_} | NOK: {nok_}")
            toplam = ok_ + nok_
            if toplam > 0:
                p_  = ok_ / toplam; r_ = 1.0 if ok_ > 0 else 0.0
                f1_ = (2 * p_ * r_ / (p_ + r_)) if (p_ + r_) > 0 else 0.0
                lbl_mtr.config(text=f"Precision: {p_:.2f} | Recall: {r_:.2f} | F1: {f1_:.2f}")
    
            target_path = self.akilli_arsivleme(file_path, is_faulty, ch_id)
            if is_faulty and target_path and self.last_conv_layer_name:
                threading.Thread(
                    target=self._async_gradcam_kaydet,
                    args=(frame_display_rgb, inp, target_path, ch_id),  
                    daemon=True
                ).start()

        except Exception as e:
            log.error(f"process_image hatasi (CH{ch_id}): {e}", exc_info=True)

    def akilli_arsivleme(self, original_path, is_faulty, ch_id):
        try:
            ts=datetime.now().strftime("%Y%m%d_%H%M%S_%f"); fn=f"CH{ch_id}_{ts}.jpg"
            td=(self.dir_ch1_nok if is_faulty else self.dir_ch1_ok) if ch_id==1 \
               else (self.dir_ch2_nok if is_faulty else self.dir_ch2_ok)
            tp=os.path.join(td,fn); shutil.move(original_path,tp)
            log.debug(f"Arsivlendi -> {tp}")
            if not is_faulty: self.klasor_limitini_koru(td,self.arsiv_limiti.get())
            return tp
        except Exception as e:
            log.warning(f"Arsivleme hatasi (CH{ch_id}): {e}"); return None

    def klasor_limitini_koru(self, folder_path, limit):
        try:
            ana=[os.path.join(folder_path,f) for f in os.listdir(folder_path)
                 if f.lower().endswith(('.jpg','.png')) and '_CAM.' not in f]
            ana.sort(key=os.path.getctime)
            for a in ana[:max(0,len(ana)-limit)]:
                try: os.remove(a)
                except Exception: pass
                cam=a.replace(".jpg","_CAM.jpg")
                if os.path.exists(cam):
                    try: os.remove(cam)
                    except Exception: pass
        except Exception as e:
            log.warning(f"Klasor limit hatasi: {e}")

    

def baslat():
    root=tk.Tk(); root.withdraw()
    splash=tk.Toplevel(root); splash.overrideredirect(True); splash.attributes('-topmost',True)
    sp=os.path.join(os.getcwd(),SPLASH_IMAGE_NAME)
    if os.path.exists(sp):
        try:
            img=Image.open(sp); img.thumbnail((600,400))
            tk_img=ImageTk.PhotoImage(img)
            lbl=tk.Label(splash,image=tk_img,bg='white'); lbl.image=tk_img; lbl.pack()
        except Exception:
            tk.Label(splash,text="MKE AI BASLATILIYOR...",font=("Arial",20)).pack(padx=50,pady=50)
    else:
        tk.Label(splash,text="MKE AI BASLATILIYOR...",font=("Arial",20)).pack(padx=50,pady=50)
    splash.update()
    w,h=splash.winfo_width(),splash.winfo_height()
    x=(splash.winfo_screenwidth()//2)-(w//2); y=(splash.winfo_screenheight()//2)-(h//2)
    splash.geometry(f"{w}x{h}+{x}+{y}")
    def loader():
        global tf; import tensorflow as tm; tf=tm
        log.info(f"TensorFlow yuklendi: {tm.__version__}")
        root.after(0,splash.destroy); root.after(0,root.deiconify)
        root.after(0,lambda:KaliteKontrolApp(root))
    threading.Thread(target=loader,daemon=True).start()
    root.mainloop()

if __name__=="__main__":
    baslat()

2026-05-17 19:53:53 [INFO] TensorFlow yuklendi: 2.20.0
2026-05-17 19:53:53 [INFO] Uygulama baslatildi.
2026-05-17 19:53:54 [INFO] Grad-CAM: 52 Conv2D bulundu, secilen: 'Conv_1'
2026-05-17 19:53:54 [INFO] Model yuklendi: C:\Users\safak\dokum_hata_tespit_modeli_MobileNet.keras | Grad-CAM: Conv_1
2026-05-17 19:53:54 [INFO] PLC ayarlari: IP=192.168.8.167 Rack=0 Slot=1 DB=1


b' TCP : Connection refused'


2026-05-17 19:53:56 [WARNING] PLC baglanti hatasi: b' TCP : Connection refused' -- Simulasyon aktif.
2026-05-17 19:54:03 [INFO] PLC ayarlari: IP=192.168.8.167 Rack=0 Slot=1 DB=1
2026-05-17 19:54:04 [INFO] PLC baglandi -> 192.168.8.167 Rack=0 Slot=1
2026-05-17 19:58:55 [INFO] Sistem baslatildi.
2026-05-17 19:59:08 [INFO] CH1 -> NOK | score=0.000 | cast_def_0_49.jpeg
2026-05-17 19:59:08 [INFO] [E2E Test] CH1 E2E Gecikme: 341.72 ms | GUNCEL ORTALAMA: 341.72 ms
2026-05-17 19:59:28 [INFO] CH2 -> OK  | score=0.994 | cast_ok_0_239.jpeg
2026-05-17 19:59:28 [INFO] [E2E Test] CH2 E2E Gecikme: 285.94 ms | GUNCEL ORTALAMA: 285.94 ms


In [4]:
print(tf.__version__)

2.20.0


In [10]:
import tensorflow as tf
print(tf.keras.__version__)

3.12.0


In [11]:
import snap7
print("Python-Snap7 Versiyonu:", snap7.__version__)



Python-Snap7 Versiyonu: 2.0.2


In [12]:
import sys

print(sys.version)

3.10.19 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 16:41:31) [MSC v.1929 64 bit (AMD64)]
